# Week2_ex1 - Straight Wire H-field

Checks the H-field around a straight current-carrying wire against the Ampere's law formula, using the EddyCurrent solver at 10Hz to basically approximate a DC field.


In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import matplotlib.pyplot as plt

In [ ]:
# 1. Ansys Electronics Desktop (AEDT) interface
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create project and set solution type
sol_type = "EddyCurrent"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. get odesign object so I can use recorded scripts
oDesign = M3D.odesign


In [ ]:
# 1. set save path for the project
# if ANSYS_PROJECT_DIR env var is set, save there, otherwise fall back to the current working dir (portable)
proj_name = "Week2_ex1"

base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

# 2. save project
proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

# 3. rename design
desi_name = "Week2_ex1"
M3D.rename_design(desi_name, save=False)


In [ ]:
# helper function to find the coil terminals
# uses the center coordinates of each face on the winding object, finds the faces with max abs(x), returns both
def find_terminal_face(winding_obj) :
    terminal_face = []

    # find maximum x position of winding object
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    #append termminal face to array
    for face in winding_obj.faces :
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001 :
            terminal_face.append(face)

    # sort
    ter_out, ter_in = sorted(terminal_face, key=lambda x : x.center[0], reverse=True)

    return ter_out, ter_in


In [ ]:
## define design variables ##

line_length = 100
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_num_seg = 12
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"


In [ ]:
# 1. copy-pasted setup code from script recording
oModule = oDesign.GetModule("AnalysisSetup")
oModule.InsertSetup("EddyCurrent", 
	[
		"NAME:Setup1",
		"Enabled:="		, True,
		[
			"NAME:MeshLink",
			"ImportMesh:="		, False
		],
		"MaximumPasses:="	, 10,
		"MinimumPasses:="	, 2,
		"MinimumConvergedPasses:=", 1,
		"PercentRefinement:="	, 15,
		"SolveFieldOnly:="	, False,
		"PercentError:="	, 1,
		"SolveMatrixAtLast:="	, True,
		"UseNonLinearIterNum:="	, False,
		"CacheSaveKind:="	, "Delta",
		"ConstantDelta:="	, "0s",
		"UseCacheFor:="		, ["Freq"],
		"UseIterativeSolver:="	, False,
		"RelativeResidual:="	, 1E-05,
		"NonLinearResidual:="	, 0.0001,
		"RelaxationFactor:="	, 1,
		"SmoothBHCurve:="	, True,
		"Frequency:="		, "10Hz",
		"HasSweepSetup:="	, False,
		"UseHighOrderShapeFunc:=", False,
		"ImportMeshForMuLink:="	, False,
		"LossAdaptiveCtrl:="	, "0.5",
		"UseMuLink:="		, False
	])

# 2. keep just the setup name instead of wrapping it with M3D.setups[-1] --
# pyaedt's automatic setup-type lookup errors out here (AEDT 2025 R2.4 reports this
# solver's solution type internally as "AC Magnetic", which pyaedt 0.15.3 doesn't
# recognize as a key -- same root cause as the Week1_ex4/ex5 create_setup() bug,
# just showing up through M3D.setups[-1] instead since this setup was made via
# script recording rather than create_setup())
setup_name = "Setup1"


In [ ]:
# 1. draw the straight wire
point_tmp = []
point_tmp.append(["-line_length/2", "0mm", "0mm"])
point_tmp.append(["line_length/2", "0mm", "0mm"])
line = M3D.modeler.create_polyline(points=point_tmp, name="line", material="copper",
                                   xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

# 2. create region and assign radiation boundary
region_tmp = ["line_length/2", "-line_length/2", "line_length", "-line_length", "line_length", "-line_length"]
region = M3D.modeler.create_region(pad_value=region_tmp ,pad_type="Absolute Position")

assignment = [region.top_face_z, region.bottom_face_z, region.top_face_y, region.bottom_face_y]  # using the 3D object Primitive's methods/attributes

# NOTE: M3D.assign_radiation() checks that solution_type == "EddyCurrent" exactly and
# raises AEDTRuntimeError("Excitation applicable only to Eddy Current.") otherwise. But
# AEDT 2025 R2 renamed this solver "AC Magnetic" internally, so self.solution_type reports
# "AC Magnetic" here -- the check fails even though this IS an eddy current solve. Bypassing
# the pyaedt wrapper and calling the raw AEDT boundary API directly sidesteps this check.
oModule = oDesign.GetModule("BoundarySetup")
face_ids = [f.id for f in assignment]
oModule.AssignRadiation(
	[
		"NAME:Radiation1",
		"Objects:=", [],
		"Faces:=", face_ids
	])

# 3. create dummy object
box_origin = ["line_diameter", "line_length", "line_length"]
box_sizes = ["-2*line_diameter", "-2*line_length", "-2*line_length"]
dummy = M3D.modeler.create_box(origin=box_origin, sizes=box_sizes, name="dummy", material="vacuum")
dummy.subtract(tool_list=line, keep_originals=True)


# 4. create a line to pull field data along
point_tmp = []
point_tmp.append(["0mm", "0mm", "0mm"])
point_tmp.append(["0mm", line_length, "0mm"])
field_line = M3D.modeler.create_polyline(points=point_tmp, name="field_line")


In [ ]:
## mesh settings ##

# NOTE: MaxLength coarsened from line_length/10 to line_length/5 -- at 10mm cells the
# dummy box (200mm x 200mm x 20mm) plus 10 adaptive passes pushed mesh size to ~68,933
# elements, over the Student license's cap. Coarser starting cells + slower per-pass
# refinement below should keep it under the cap.

oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:line_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["line"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/5",
	])

oModule.AssignLengthOp(
	[
		"NAME:dummy_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["dummy"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/5"
	])


In [ ]:
# 1. set the terminal faces of the coil object
ter_out, ter_in = find_terminal_face(line)  # call the helper to find terminal faces

M3D.assign_coil(assignment=ter_out, conductors_number=1, polarity="Negative", name="Out")
M3D.assign_coil(assignment=ter_in, conductors_number=1, polarity="Positive", name="In")


# 2. create a winding and add the coils to it
coil = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=True, current="Current", 
                            resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, 
                             name="coil")

M3D.add_winding_coils(coil.name, coils=["Out", "In"])


In [ ]:
## analyze ##

oDesign.Analyze(setup_name)

In [ ]:
## plot report ##

oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport("Calculator Expressions Plot 1", "Fields", "Rectangular Plot", "Setup1 : LastAdaptive", 
	[
		"Context:="		, "field_line",
		"PointCount:="		, 301
	], 
	[
		"Distance:="		, ["All"],
		"Freq:="		, ["All"],
		"Phase:="		, ["0deg"],
		"line_length:="		, ["Nominal"],
		"line_diameter:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "Distance",
		"Y Component:="		, ["Mag_H"]
	])


csv_name = f"{proj_name}_H_field_result.csv"
dir_csv = os.path.join(dir, csv_name)
oModule.ExportToFile("Calculator Expressions Plot 1", dir_csv, False)



In [ ]:
## plot in python ##


# 1) read the CSV
#    - has a header, assuming first column is 'Distance (mm)' and second is 'B (T)'
df = pd.read_csv(dir_csv)

# 2) pull out distance (mm) and field (T) columns from the CSV
distance_mm = df['Distance [mm]'].values
H_data      = df['Mag_H [kA_per_meter]'].values

# 3) mm -> m
r_data = distance_mm * 1e-3

# 4) constants and wire params for the theoretical field
mu0 = 4.0 * np.pi * 1e-7  # [H/m] permeability of free space
I   = Current               # [A]  current
R   = 0.005               # [m]  wire radius (e.g. 1 mm)

# 5) field equations inside/outside the wire
#    - inside (r <= R): B_inner = (mu0 * r * I) / (2π R^2)
#    - outside (r >  R): B_outer = (mu0 * I)     / (2π r)
def H_theory(r_array):
    return np.where(
        r_array <= R,
        r_array * I / (2.0 * np.pi * R**2),  # inside
        I           / (2.0 * np.pi * r_array) # outside
    )

# calculate theoretical values
H_calc = H_theory(r_data)
H_calc = H_calc*1e-3    # convert to [kA/m]

# 6) plot
plt.figure(figsize=(8,6))
plt.plot(distance_mm, H_data, 'ro-',  label='Ansys simulation')
plt.plot(distance_mm, H_calc, 'b--', label='formula')
plt.xlabel('distance (mm)')
plt.ylabel('H (kA/m)')
plt.title(' ')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
## save project ##

M3D.save_project()
